# bulk setup of users.
* copy constants.py.example (rename to constants.py) and paste the admin token
* copy example.csv, rename and and fill with data
* input file name below
* run all cells. the output will  be in "created_users.csv")

### install requred libraries

In [ ]:
%pip install loguru

In [ ]:
%pip install pandas

### setup

In [ ]:
import requests
from loguru import logger
import pandas as pd
from constants import ADMIN_TOKEN

In [ ]:
# for local use http://127.0.0.1:5000
API_BASE_URL = "https://api.meissnerm.eu"

In [ ]:
INPUT_FILE = ""

### read and check input file

In [ ]:
df = pd.read_csv(INPUT_FILE, dtype={'first_name': "string", 'last_name': "string", 'language': "string"})
df

In [ ]:
assert list(df.columns) == ["first_name", "last_name", "language"]
assert df['first_name'].notna().all()
assert df['last_name'].notna().all()

### start validate session

In [ ]:
health_response = requests.get(f"{API_BASE_URL}/health")

logger.info(health_response.json())
health_response.raise_for_status()

In [ ]:
session = requests.Session()

In [ ]:
login_response = session.post(f"{API_BASE_URL}/auth/login", json ={"token": C.ADMIN_TOKEN})

logger.info(login_response.json())
login_response.raise_for_status()

In [ ]:
# hack to access localhost (no https)
# 
if API_BASE_URL == "http://127.0.0.1:5000":
    for cookie in session.cookies:
        if cookie.name == 'session_token':
            cookie.secure = False  # Remove the secure flag

### bulk setup users

In [ ]:
df["token"] = None

In [ ]:
try{
    for i, row in df.iterrows():
    print(f"creating {row.first_name} / {row.last_name} / {row.language}")
    
    create_user_response = session.post(f"{API_BASE_URL}/user", json ={"first_name": row.first_name, "last_name": row.last_name, "role": "USER", "language": None if row.language is pd.NA else row.language})
    create_user_response.raise_for_status()
    df.loc[i, "token"] = create_user_response.json()["token"]
df.to_csv('tools/created_users.csv')
df